## **Reproducing and Extending a Dynamical Model of Dopamine Regulation During Levodopa Therapy**

### **Student name:** Nahal Habibizadeh 
### **Student ID:** 20942866

**Description:**
Dopaminergic Terminal Model Implementation
Based on Best et al. (2009) and Reed et al. (2012)

Reproduces:
1. Steady-state homeostasis of the DA terminal
2. Passive stabilization of extracellular DA (Fig 6, Reed 2012)
3. Extracellular DA time courses during LD dosing for varying f (Fig 7A, Reed 2012)
4. Therapeutic time window vs f (Fig 7C, Reed 2012)

Novel Extension:
5. Coupling extracellular DA to a simplified neuronal excitability model

In [2]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os


### Model Parameters

All values taken from Best et al. (2009) Table 2 and Reed et al. (2012) Tables A2–A5.

In [ ]:
# Best et al. 2009 — tyrosine hydroxylase
th_vmax  = 125.0
th_ktyr  = 46.0
th_kbh4  = 60.0
th_ki_c  = 110.0
th_ki_t  = 160.0

# AADC
adc_vmax = 10000.0
adc_km   = 130.0

# VMAT
vmat_vmax = 7082.0
vmat_km   = 3.0
vmat_leak = 40.0

# DAT
dat_vmax = 8000.0
dat_km   = 0.2

# DRR (biopterin recycling)
drr_vf   = 5000.0
drr_vb   = 80.0
drr_kbh2 = 100.0
drr_knadph = 75.0
drr_kbh4 = 10.0
drr_knadp = 75.0
nadph = 100.0
nadp  = 100.0

# Tyrosine transport + pool
btyr       = 97.0
tyr_vmax   = 400.0
tyr_km     = 64.0
k_pool_fwd = 6.0
k_pool_rev = 0.6

# Catabolism / removal
kcat_tyr  = 0.2
kcat_cda  = 10.0
ecat_vmax = 30.0
ecat_km   = 3.0
kcat_hva  = 3.45
kcat_pool = 0.2
k_rem0    = 400.0

fr = 1.0  # tonic firing rate (normalised to 1)

### Reaction Velocities

The dopamine pathway: tyrosine → L-DOPA (TH, rate-limiting) → cytosolic DA (AADC) → vesicular DA (VMAT) → extracellular DA (release) → reuptake (DAT).

In [ ]:
def tyr_import(bld):
    return tyr_vmax * bld / (tyr_km + bld)

def th_rate(tyr, bh4, cda, eda):
    si   = 0.56 / (1.0 + (tyr / th_ki_t)**2)
    en   = 0.002
    rat  = eda / (en + 1e-12)
    auto = (4.5 / (1.0 + rat**4)) + 0.5
    mm   = th_vmax * tyr * bh4 / ((tyr + th_ktyr) * (bh4 + th_kbh4) * (1.0 + cda / th_ki_c))
    return si * auto * mm

def drr_rate(bh2, bh4):
    fwd = drr_vf * bh2 * nadph / ((drr_kbh2 + bh2) * (drr_knadph + nadph))
    bwd = drr_vb * bh4 * nadp  / ((drr_kbh4  + bh4) * (drr_knadp  + nadp))
    return fwd - bwd

def aadc_rate(ld):
    return adc_vmax * ld / (adc_km + ld)

def vmat_rate(cda, vda):
    return vmat_vmax * cda / (vmat_km + cda) - vmat_leak * vda

def dat_rate(eda):
    return dat_vmax * eda / (dat_km + eda)

def eda_catab(eda):
    return ecat_vmax * eda / (ecat_km + eda)


In [5]:
####################################################
# LD DOSE FUNCTION (from Reed 2012, Figure 2A)
####################################################

def ld_dose_serum(t, t_dose=5.0, peak_conc=100.0, half_life=1.5):
    """
    Serum LD concentration during a dose.
    Rises quickly after administration and decays with ~90 min half-life.
    t_dose: time of dose administration (hours)
    peak_conc: peak serum concentration (uM)
    half_life: half-life in hours (~1.5 hr)
    """
    if t < t_dose:
        return 0.0
    dt = t - t_dose
    k_decay = np.log(2) / half_life
    k_rise = 5.0  # fast absorption
    return peak_conc * (np.exp(-k_decay * dt) - np.exp(-k_rise * dt)) * k_rise / (k_rise - k_decay)


### SIMULATION 1: Verify Steady State

In [6]:
ss = get_steady_state(f=1.0)
names = ['bh2', 'bh4', 'tyr', 'ldopa', 'cda', 'vda', 'eda', 'hva', 'tyrpool']
expected = [41.0, 319.0, 126.0, 0.36, 2.65, 81.0, 0.002, 7.68, 952.0]

print(f"\n{'Variable':<12} {'Model SS':>12} {'Expected':>12}")
print("-" * 40)
for name, val, exp in zip(names, ss, expected):
    print(f"{name:<12} {val:>12.4f} {exp:>12.4f}")
ss


Variable         Model SS     Expected
----------------------------------------
bh2                3.2571      41.0000
bh4              356.7429     319.0000
tyr              114.9038     126.0000
ldopa              0.5963       0.3600
cda                4.4596       2.6500
vda              103.2650      81.0000
eda                0.0026       0.0020
hva               12.9339       7.6800
tyrpool          861.7788     952.0000


array([3.25711894e+00, 3.56742881e+02, 1.14903835e+02, 5.96266890e-01,
       4.45961454e+00, 1.03264965e+02, 2.58816103e-03, 1.29339144e+01,
       8.61778763e+02])

### SIMULATION 2: Passive Stabilization (Reed 2012, Fig 6)
Reproduce Figure 6 from Reed et al. (2012):
    Extracellular DA concentration as a function of fraction of SNc cells alive.

In [8]:


f_values = np.concatenate([
    np.arange(0.01, 0.05, 0.01),
    np.arange(0.05, 0.2, 0.02),
    np.arange(0.2, 1.01, 0.05)
])

eda_values = []

for f in f_values:
    ss = get_steady_state(f=f)
    eda_ss = ss[6]  # eda is index 6
    eda_values.append(eda_ss)
    print(f"  f = {f:.3f}, eda = {eda_ss:.6f} uM = {eda_ss*1000:.4f} nM")

# Normalize to f=1 value
eda_normal = eda_values[-1]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(f_values, np.array(eda_values) * 1000, 'b-o', markersize=4, linewidth=2)
ax.set_xlabel('Fraction of SNc cells alive (f)', fontsize=13)
ax.set_ylabel('Extracellular DA concentration (nM)', fontsize=13)
ax.set_title('Passive Stabilization of Extracellular DA\n(cf. Reed et al. 2012, Figure 6)', fontsize=14)
ax.set_xlim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig(f"fig_passive_stabilization.png", dpi=200, bbox_inches='tight')
plt.close()
print(f"\n  Saved: fig_passive_stabilization.png")

f_values, eda_values

  f = 0.010, eda = 0.001438 uM = 1.4384 nM
  f = 0.020, eda = 0.001844 uM = 1.8443 nM
  f = 0.030, eda = 0.002029 uM = 2.0288 nM
  f = 0.040, eda = 0.002138 uM = 2.1384 nM
  f = 0.050, eda = 0.002212 uM = 2.2123 nM
  f = 0.070, eda = 0.002307 uM = 2.3066 nM
  f = 0.090, eda = 0.002365 uM = 2.3646 nM
  f = 0.110, eda = 0.002404 uM = 2.4041 nM
  f = 0.130, eda = 0.002433 uM = 2.4328 nM
  f = 0.150, eda = 0.002455 uM = 2.4545 nM
  f = 0.170, eda = 0.002472 uM = 2.4716 nM
  f = 0.190, eda = 0.002485 uM = 2.4854 nM
  f = 0.200, eda = 0.002491 uM = 2.4913 nM
  f = 0.250, eda = 0.002514 uM = 2.5144 nM
  f = 0.300, eda = 0.002530 uM = 2.5301 nM
  f = 0.350, eda = 0.002542 uM = 2.5416 nM
  f = 0.400, eda = 0.002550 uM = 2.5503 nM
  f = 0.450, eda = 0.002557 uM = 2.5572 nM
  f = 0.500, eda = 0.002563 uM = 2.5627 nM
  f = 0.550, eda = 0.002567 uM = 2.5673 nM
  f = 0.600, eda = 0.002571 uM = 2.5711 nM
  f = 0.650, eda = 0.002574 uM = 2.5744 nM
  f = 0.700, eda = 0.002577 uM = 2.5771 nM
  f = 0.750

(array([0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.09, 0.11, 0.13, 0.15, 0.17,
        0.19, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55, 0.6 , 0.65,
        0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95, 1.  ]),
 [np.float64(0.0014383777415793407),
  np.float64(0.0018443090016058827),
  np.float64(0.0020287610176015066),
  np.float64(0.0021384269529082832),
  np.float64(0.002212313251057861),
  np.float64(0.002306585152067659),
  np.float64(0.0023646364368096803),
  np.float64(0.0024041289101606857),
  np.float64(0.002432782612652647),
  np.float64(0.002454538880672318),
  np.float64(0.0024716283400434366),
  np.float64(0.0024854108178675558),
  np.float64(0.002491348758506864),
  np.float64(0.002514367692591811),
  np.float64(0.00253013025873941),
  np.float64(0.0025416027754104533),
  np.float64(0.002550327769729381),
  np.float64(0.00255718709112962),
  np.float64(0.002562721545465084),
  np.float64(0.002567281276779),
  np.float64(0.0025711030049720453),
  np.float64(0.0025743525223332186),
  

### ODE System

Nine-variable system: `[bh2, bh4, tyr, ldopa, cda, vda, eda, hva, tyrpool]`.  
Parameter `f` is the fraction of SNc cells still alive (1 = healthy, → 0 = advanced PD).  
The removal rate scales as `k_rem = k_rem0 / f` — as terminal density falls, released DA is increasingly likely to diffuse out rather than be recaptured.

In [ ]:
def da_odes(t, y, f=1.0, ld_func=None):
    bh2, bh4, tyr, ld, cda, vda, eda, hva, tpool = [max(v, 0) for v in y]

    vt   = tyr_import(btyr)
    vth  = th_rate(tyr, bh4, cda, eda)
    vdrr = drr_rate(bh2, bh4)
    va   = aadc_rate(ld)
    vm   = vmat_rate(cda, vda)
    vd   = dat_rate(eda)
    ve   = eda_catab(eda)

    ld_in = 0.0
    if ld_func is not None:
        sld  = ld_func(t)
        styr, stryp = 63.0, 82.0
        denom = 1.0 + styr/64.0 + stryp/15.0
        ld_in = 1840.0 * sld / (32.0 * denom + sld)

    kr = k_rem0 / max(f, 0.001)

    return [
        vth - vdrr,
        vdrr - vth,
        vt - vth - k_pool_fwd*tyr + k_pool_rev*tpool - kcat_tyr*tyr,
        vth - va + ld_in,
        va - vm + vd - kcat_cda*cda,
        vm - fr*vda,
        fr*vda - vd - ve - kr*eda,
        kcat_cda*cda + ve - kcat_hva*hva,
        k_pool_fwd*tyr - k_pool_rev*tpool - kcat_pool*tpool
    ]

def to_ss(f=1.0):
    y0  = [41.0, 319.0, 126.0, 0.36, 2.65, 81.0, 0.002, 7.68, 952.0]
    sol = solve_ivp(da_odes, [0, 200], y0, args=(f, None),
                    method='BDF', rtol=1e-8, atol=1e-10, max_step=0.5)
    return sol.y[:, -1]


In [11]:
# SIMULATION 4: Therapeutic Time Window (Reed 2012, Fig 7C)
def run_therapeutic_window():
    """
    Reproduce Figure 7C from Reed et al. (2012):
    Therapeutic time window as a function of f.
    """
    print("\n" + "=" * 60)
    print("SIMULATION 4: Therapeutic Time Window vs. f")
    print("=" * 60)
    
    f_values = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
    t_span = [0, 25]
    t_eval = np.linspace(0, 25, 3000)
    t_dose = 5.0
    
    # Get baselines for threshold
    ss_normal = get_steady_state(f=1.0)
    ss_02 = get_steady_state(f=0.2)
    threshold = (ss_normal[6] + ss_02[6]) / 2.0
    
    windows = []
    
    for f in f_values:
        print(f"  Computing for f = {f}...")
        ss = get_steady_state(f=f)
        
        def ld_func(t, t_dose=t_dose):
            return ld_dose_serum(t, t_dose=t_dose, peak_conc=100.0, half_life=1.5)
        
        sol = solve_ivp(da_terminal_odes, t_span, ss,
                       args=(f, ld_func),
                       method='BDF', rtol=1e-8, atol=1e-10,
                       t_eval=t_eval, max_step=0.05)
        
        eda = sol.y[6]
        above_threshold = eda > threshold
        
        # Find duration above threshold
        if np.any(above_threshold):
            indices = np.where(above_threshold)[0]
            t_start = sol.t[indices[0]]
            t_end = sol.t[indices[-1]]
            window = t_end - t_start
        else:
            window = 0.0
        
        windows.append(window)
        print(f"    Therapeutic window: {window:.2f} hours")
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.plot(f_values, windows, 'ko-', markersize=8, linewidth=2)
    ax.set_xlabel('Fraction of SNc cells alive (f)', fontsize=13)
    ax.set_ylabel('Therapeutic window (hours)', fontsize=13)
    ax.set_title('Therapeutic Time Window vs. Disease Progression\n(cf. Reed et al. 2012, Figure 7C)', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)
    
    plt.tight_layout()
    plt.savefig(f"fig_therapeutic_window.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: fig_therapeutic_window.png")
    
    return f_values, windows


In [12]:
# SIMULATION 5: Autoreceptor Homeostasis (Best 2009, Figure 9 analog)
def run_autoreceptor_homeostasis():
    """
    Show the homeostatic effect of autoreceptors on TH velocity 
    and extracellular DA as firing rate changes.
    """
    print("\n" + "=" * 60)
    print("SIMULATION 5: Autoreceptor Homeostasis")
    print("=" * 60)
    
    # Vary firing rate from 60% to 200% of normal
    fire_fracs = np.linspace(0.6, 2.0, 20)
    
    eda_with_auto = []
    eda_no_auto = []
    
    for frac in fire_fracs:
        # Modify fire_rate globally for this simulation
        # We do a simplified steady-state calculation
        
        # With autoreceptors (normal model)
        y0 = [41.0, 319.0, 126.0, 0.36, 2.65, 81.0, 0.002, 7.68, 952.0]
        
        def ode_with_auto(t, y):
            return da_terminal_odes(t, y, f=1.0)
        
        # Temporarily modify fire_rate
        global fire_rate
        fire_rate = frac  # Scale firing rate
        
        sol = solve_ivp(ode_with_auto, [0, 100], y0,
                       method='BDF', rtol=1e-8, atol=1e-10, max_step=1.0)
        eda_with_auto.append(sol.y[6, -1] * 1000)  # nM
    
    fire_rate = 1.0  # Reset
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.plot(fire_fracs * 100, eda_with_auto, 'r-o', markersize=5, linewidth=2,
            label='With autoreceptors')
    ax.set_xlabel('Firing rate (% of normal)', fontsize=13)
    ax.set_ylabel('Extracellular DA (nM)', fontsize=13)
    ax.set_title('Autoreceptor Homeostasis:\nExtracellular DA vs. Firing Rate\n(cf. Best et al. 2009, Figure 9B)',
                fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)
    
    plt.tight_layout()
    plt.savefig(f"fig_autoreceptor_homeostasis.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: fig_autoreceptor_homeostasis.png")

In [16]:
def run_neural_coupling_extension():
    """
    Novel extension: Couple extracellular DA dynamics to a simplified
    neuronal excitability model.
    
    Model: tau * dV/dt = -V + g(D_ext) * I_input
    where g(D) = g0 * (1 + alpha * D / (K_D + D))
    
    This creates a dopamine-modulated gain that amplifies neural responses.
    We examine how degeneration-driven DA pulses affect neural stability.
    """
    print("\n" + "=" * 60)
    print("NOVEL EXTENSION: DA-Modulated Neural Excitability")
    print("=" * 60)
    
    # Parameters for the neural model
    tau_n = 0.02       # membrane time constant (hours) ~ 72 seconds
    g0 = 1.0           # baseline gain
    alpha = 2.0        # DA modulation strength
    K_D = 2.0          # nM, half-max DA concentration for gain modulation
    I_base = 1.0       # baseline input current (normalized)
    
    def gain_function(D_nM):
        """Dopamine-dependent gain function."""
        return g0 * (1.0 + alpha * D_nM / (K_D + D_nM))
    
    # First, get DA time courses for different f values during LD dose
    f_values = [0.2, 0.1, 0.05, 0.01]
    colors = ['blue', 'green', 'magenta', 'cyan']
    labels = ['f = 0.2', 'f = 0.1', 'f = 0.05', 'f = 0.01']
    
    t_span = [0, 20]
    t_eval = np.linspace(0, 20, 5000)
    t_dose = 5.0
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    
    for f, color, label in zip(f_values, colors, labels):
        print(f"  Running neural coupling for f = {f}...")
        
        ss = get_steady_state(f=f)
        
        def ld_func(t, t_dose=t_dose):
            return ld_dose_serum(t, t_dose=t_dose, peak_conc=100.0, half_life=1.5)
        
        sol = solve_ivp(da_terminal_odes, t_span, ss,
                       args=(f, ld_func),
                       method='BDF', rtol=1e-8, atol=1e-10,
                       t_eval=t_eval, max_step=0.05)
        
        eda_nM = sol.y[6] * 1000  # Convert to nM
        t_arr = sol.t
        
        # Compute gain over time
        gain_t = np.array([gain_function(d) for d in eda_nM])
        
        # Simulate neural response: tau * dV/dt = -V + g(D)*I
        # For a constant input I_base, the instantaneous equilibrium is V* = g(D)*I
        # We solve the full ODE for realism
        V = np.zeros_like(t_arr)
        V[0] = gain_function(eda_nM[0]) * I_base  # Start at equilibrium
        
        for i in range(1, len(t_arr)):
            dt = t_arr[i] - t_arr[i-1]
            dVdt = (-V[i-1] + gain_function(eda_nM[i]) * I_base) / tau_n
            V[i] = V[i-1] + dVdt * dt
        
        # Plot DA
        axes[0].plot(t_arr, eda_nM, color=color, linewidth=1.5, label=label)
        # Plot gain
        axes[1].plot(t_arr, gain_t, color=color, linewidth=1.5, label=label)
        # Plot neural response
        axes[2].plot(t_arr, V, color=color, linewidth=1.5, label=label)
    
    axes[0].set_ylabel('Extracellular DA (nM)', fontsize=12)
    axes[0].set_title('Dopamine-Modulated Neural Excitability During LD Dose', fontsize=14)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_ylabel('Gain g(D)', fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=9)
    
    axes[2].set_ylabel('Neural response V(t)', fontsize=12)
    axes[2].set_xlabel('Time (hours)', fontsize=12)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(fontsize=9)
    
    for ax in axes:
        ax.set_xlim([4, 18])
        ax.tick_params(labelsize=10)
    
    plt.tight_layout()
    plt.savefig(f"fig_neural_coupling.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: fig_neural_coupling.png")
    
    # --- Additional analysis: Peak gain and gain variance vs f ---
    print("\n  Computing peak gain and gain variability vs f...")
    
    f_scan = np.array([0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.5, 1.0])
    peak_gains = []
    gain_ranges = []
    
    for f in f_scan:
        ss = get_steady_state(f=f)
        
        def ld_func(t, t_dose=t_dose):
            return ld_dose_serum(t, t_dose=t_dose, peak_conc=100.0, half_life=1.5)
        
        sol = solve_ivp(da_terminal_odes, t_span, ss,
                       args=(f, ld_func),
                       method='BDF', rtol=1e-8, atol=1e-10,
                       t_eval=t_eval, max_step=0.05)
        
        eda_nM = sol.y[6] * 1000
        gains = np.array([gain_function(d) for d in eda_nM])
        
        peak_gains.append(np.max(gains))
        gain_ranges.append(np.max(gains) - np.min(gains))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.plot(f_scan, peak_gains, 'ro-', markersize=6, linewidth=2)
    ax1.set_xlabel('Fraction of SNc cells alive (f)', fontsize=12)
    ax1.set_ylabel('Peak neural gain during LD dose', fontsize=12)
    ax1.set_title('Peak DA-Modulated Gain\nvs. Disease Progression', fontsize=13)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(f_scan, gain_ranges, 'bs-', markersize=6, linewidth=2)
    ax2.set_xlabel('Fraction of SNc cells alive (f)', fontsize=12)
    ax2.set_ylabel('Gain range (max - min)', fontsize=12)
    ax2.set_title('Gain Variability During LD Dose\nvs. Disease Progression', fontsize=13)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"fig_gain_analysis.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: fig_gain_analysis.png")

In [17]:
# SUBSTRATE INHIBITION DEMONSTRATION (Best 2009, Figure 3 analog)

def run_substrate_inhibition():
    """
    Demonstrate the effect of substrate inhibition of TH by tyrosine
    on stabilizing vesicular dopamine during meal-driven tyrosine fluctuations.
    """
    print("\n" + "=" * 60)
    print("SIMULATION 6: Substrate Inhibition of TH")
    print("=" * 60)
    
    # Plot TH velocity vs tyrosine concentration
    tyr_range = np.linspace(1, 500, 500)
    bh4_ss = 319.0
    cda_ss = 2.65
    eda_ss = 0.002
    
    # With substrate inhibition (normal)
    v_with_si = []
    for t in tyr_range:
        v = V_TH(t, bh4_ss, cda_ss, eda_ss)
        v_with_si.append(v)
    
    # Without substrate inhibition
    v_no_si = []
    for t in tyr_range:
        MM = TH_Vmax * t * bh4_ss / ((t + TH_Ktyr) * (bh4_ss + TH_Kbh4) * (1.0 + cda_ss / TH_Ki_cda))
        auto = 1.0  # At normal eda
        v_no_si.append(auto * MM)
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.plot(tyr_range, v_with_si, 'b-', linewidth=2, label='With substrate inhibition')
    ax.plot(tyr_range, v_no_si, 'r-', linewidth=2, label='Without substrate inhibition')
    ax.axvline(x=126, color='gray', linestyle='--', alpha=0.5, label='Normal tyr (126 μM)')
    ax.set_xlabel('Tyrosine concentration (μM)', fontsize=13)
    ax.set_ylabel('Velocity of TH reaction (μM/hr)', fontsize=13)
    ax.set_title('Substrate Inhibition of Tyrosine Hydroxylase\n(cf. Best et al. 2009, Figure 2)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=11)
    
    plt.tight_layout()
    plt.savefig(f"fig_substrate_inhibition.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: fig_substrate_inhibition.png")



In [21]:
 # 1. Verify steady state
ss = run_steady_state_verification()
    
    # 2. Substrate inhibition
run_substrate_inhibition()
    
    # 3. Autoreceptor homeostasis
run_autoreceptor_homeostasis()
    
    # 4. Passive stabilization
run_passive_stabilization()
    
    # 5. LD dose time courses
run_ld_dose_time_courses()
    
    # 6. Therapeutic time window
run_therapeutic_window()
    
    # 7. Novel extension: neural coupling
run_neural_coupling_extension()


SIMULATION 1: Steady-State Verification

Variable         Model SS     Expected
----------------------------------------
bh2                3.2571      41.0000
bh4              356.7429     319.0000
tyr              114.9038     126.0000
ldopa              0.5963       0.3600
cda                4.4596       2.6500
vda              103.2650      81.0000
eda                0.0026       0.0020
hva               12.9339       7.6800
tyrpool          861.7788     952.0000

SIMULATION 6: Substrate Inhibition of TH

  Saved: fig_substrate_inhibition.png

SIMULATION 5: Autoreceptor Homeostasis

  Saved: fig_autoreceptor_homeostasis.png

SIMULATION 2: Passive Stabilization of Extracellular DA
  f = 0.010, eda = 0.001438 uM = 1.4384 nM
  f = 0.020, eda = 0.001844 uM = 1.8443 nM
  f = 0.030, eda = 0.002029 uM = 2.0288 nM
  f = 0.040, eda = 0.002138 uM = 2.1384 nM
  f = 0.050, eda = 0.002212 uM = 2.2123 nM
  f = 0.070, eda = 0.002307 uM = 2.3066 nM
  f = 0.090, eda = 0.002365 uM = 2.3646 nM
  f = 